In [1]:
her2st_folder = '../data/her2st/data'
stnet_folder = '../data/ST/BC'
skin_folder = '../data/skin'
pcw_folder = '../data/ST-images/PCW'
mouse_folder = '../data/ST-images/MOUSE'

her2st_imgs_folder = her2st_folder + '/ST-imgs'
her2st_st_postion = her2st_folder + '/ST-spotfiles'
her2st_gene_expression = her2st_folder + '/ST-cnts'
print(her2st_imgs_folder)
print(her2st_st_postion)
print(her2st_gene_expression)

stnet_imgs_folder = stnet_folder + '/ST-imgs'
stnet_st_postion = stnet_folder + '/ST-spotfiles'
stnet_gene_expression = stnet_folder + '/ST-cnts'
print(stnet_imgs_folder)
print(stnet_st_postion)
print(stnet_gene_expression)

skin_imgs_folder = skin_folder + '/ST-imgs'
skin_st_postion = skin_folder + '/ST-spotfiles'
skin_gene_expression = skin_folder + '/ST-cnts'
print(skin_imgs_folder)
print(skin_st_postion)
print(skin_gene_expression)

pcw_imgs_folder =  pcw_folder + '/ST-imgs'
pcw_st_postion = pcw_folder + '/ST-spotfiles'
pcw_gene_expression = pcw_folder + '/ST-cnts'
print(pcw_imgs_folder)
print(pcw_st_postion)
print(pcw_gene_expression)

mouse_imgs_folder =  mouse_folder + '/ST-imgs'
mouse_st_postion = mouse_folder + '/ST-spotfiles'
mouse_gene_expression = mouse_folder + '/ST-cnts'
print(mouse_imgs_folder)
print(mouse_st_postion)
print(mouse_gene_expression)

In [ ]:
import os
import pandas as pd

genes = []

for i in os.listdir(her2st_gene_expression):
    path = os.path.join(her2st_gene_expression,i)
    print(f"Gene expression file : {path}")
    data = pd.read_csv(path, sep='\t')
    spots = data[data.columns[0]].to_list()
    # genes_i = data.iloc[0].to_list()
    genes_i = data.columns.tolist()[1:]
    genes.append(genes_i)
    print(spots)
    print(len(spots))
    print(genes_i)
    print(len(genes_i))

In [ ]:
total_genes = set().union(*genes)
lens = len(total_genes)
print(f"{lens}")

inter_genes = total_genes.intersection(*genes)
lens2 = len(inter_genes)
print(f"{lens2}")

In [28]:
gene_expression_sum = {}
for filename in os.listdir(her2st_gene_expression):
    if filename.endswith('.tsv'):
        file_path = os.path.join(her2st_gene_expression, filename)
        print(f"Processing file: {file_path}")
        df = pd.read_csv(file_path, sep='\t')
        genes = df.columns[1:]
        print(genes)
        for gene in genes:
            total_expression = df[gene].sum()
            if gene in gene_expression_sum:
                gene_expression_sum[gene] += total_expression
            else:
                gene_expression_sum[gene] = total_expression
print(gene_expression_sum)

In [ ]:
sorted_gene_expression = sorted(gene_expression_sum.items(), key=lambda x: x[1], reverse=True)
gene_select_heg_nums = []
with open('her2st_top_250_genes.csv', 'w') as f:
    f.write("Gene,Expression\n")
    for gene, expression in sorted_gene_expression[:500]:
        if gene in inter_genes:
            gene_select_heg_nums.append(gene)
            f.write(f"{gene},{expression}\n")
        if len(gene_select_heg_nums) == 250:
            break

print("Top 250 genes have been saved to 'her2st_top_250_genes.csv'.")

In [ ]:
all_hvgs = []

for filename in os.listdir(her2st_gene_expression):
    if filename.endswith('.tsv'): 
        file_path = os.path.join(her2st_gene_expression, filename)
        print(f"Processing file: {file_path}")
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        gene_variance = df.var(axis=0)
        top_100_hvgs = gene_variance.sort_values(ascending=False).head(100)
        all_hvgs.extend(top_100_hvgs.index.tolist())



In [ ]:
from collections import Counter
gene_counter = Counter(all_hvgs)
top_50_common_genes = [gene for gene, count in gene_counter.most_common() if gene in inter_genes][:50]
print("Top 50 common genes across all files:")
print(top_50_common_genes)
print(len(top_50_common_genes))
all_right = True
for i in top_50_common_genes:
    if i not in inter_genes:
        all_right = False
assert(all_right == True)
df = pd.DataFrame(top_50_common_genes, columns=['Gene'])
df.to_csv('her2st_top_50_hvg_genes.csv', index=False)

print("Top 50 common genes have been saved to 'her2st_top_50_hvg_genes.csv'.")


In [ ]:
output_dir = './her2st_top250_heg'
selected_genes_file = 'her2st_top_250_genes.csv'
os.makedirs(output_dir, exist_ok=True)
selected_genes_df = pd.read_csv(selected_genes_file)
selected_genes = selected_genes_df['Gene'].tolist() 
for filename in os.listdir(her2st_gene_expression):
    if filename.endswith('.tsv'): 
        file_path = os.path.join(her2st_gene_expression, filename)
        print(f"Processing file: {file_path}")
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        filtered_df = df[selected_genes]
        output_file = os.path.join(output_dir, filename)
        filtered_df.to_csv(output_file, sep='\t', index=True)
        print(f"Filtered file saved to: {output_file}")

In [ ]:
output_dir = './her2st_top50_hvg'
selected_genes_file = 'her2st_top_50_hvg_genes.csv'
os.makedirs(output_dir, exist_ok=True)
selected_genes_df = pd.read_csv(selected_genes_file)
selected_genes = selected_genes_df['Gene'].tolist()  
for filename in os.listdir(her2st_gene_expression):
    if filename.endswith('.tsv'): 
        file_path = os.path.join(her2st_gene_expression, filename)
        print(f"Processing file: {file_path}")
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        filtered_df = df[selected_genes]
        output_file = os.path.join(output_dir, filename)
        filtered_df.to_csv(output_file, sep='\t', index=True)
        print(f"Filtered file saved to: {output_file}")

In [ ]:
for i in os.listdir('./'):
    if i.endswith('.csv'):
        os.remove(i)

In [ ]:
import os
import pandas as pd
print(stnet_imgs_folder)
print(stnet_st_postion)
print(stnet_gene_expression)

genes = []

for i in os.listdir(stnet_gene_expression):
    path = os.path.join(stnet_gene_expression,i)
    print(f"Gene expression file : {path}")
    data = pd.read_csv(path, sep='\t')
    spots = data[data.columns[0]].to_list()
    genes_i = data.columns.tolist()[1:]
    genes.append(genes_i)
    print(spots)
    print(len(spots))
    print(genes_i)
    print(len(genes_i))

In [ ]:
import os
import pandas as pd

genes = []

for i in os.listdir(stnet_gene_expression):
    path = os.path.join(stnet_gene_expression,i)
    print(f"Gene expression file : {path}")
    data = pd.read_csv(path, sep='\t')
    spots = data[data.columns[0]].to_list()
    genes_i = data.columns.tolist()[1:]
    genes.append(genes_i)
    print(spots)
    print(len(spots))
    print(genes_i)
    print(len(genes_i))

In [ ]:
total_genes = set().union(*genes)
lens = len(total_genes)

inter_genes = total_genes.intersection(*genes)
lens2 = len(inter_genes)


In [13]:
gene_expression_sum = {}
for filename in os.listdir(stnet_gene_expression):
    if filename.endswith('.tsv'):
        file_path = os.path.join(stnet_gene_expression, filename)
        print(f"Processing file: {file_path}")
        df = pd.read_csv(file_path, sep='\t')
        temp_genes = df.columns[1:]
        print(temp_genes)
        for gene in inter_genes:
            total_expression = df[gene].sum()
            if gene in gene_expression_sum:
                gene_expression_sum[gene] += total_expression
            else:
                gene_expression_sum[gene] = total_expression

print(gene_expression_sum)

In [ ]:
sorted_gene_expression = sorted(gene_expression_sum.items(), key=lambda x: x[1], reverse=True)
gene_select_heg_nums = []
with open('stnet_top_250_genes.csv', 'w') as f:
    f.write("Gene,Expression\n")
    for gene, expression in sorted_gene_expression[:500]:
        if gene in inter_genes:
            gene_select_heg_nums.append(gene)
            f.write(f"{gene},{expression}\n")
        if len(gene_select_heg_nums) == 250:
            break

print("Top 250 genes have been saved to 'stnet_top_250_genes.csv'.")

In [ ]:
output_dir = './stnet_top250_heg'
selected_genes_file = 'stnet_top_250_genes.csv'
os.makedirs(output_dir, exist_ok=True)
selected_genes_df = pd.read_csv(selected_genes_file)
selected_genes = selected_genes_df['Gene'].tolist() 
for filename in os.listdir(stnet_gene_expression):
    if filename.endswith('.tsv'): 
        file_path = os.path.join(stnet_gene_expression, filename)
        print(f"Processing file: {file_path}")
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        filtered_df = df[selected_genes]
        output_file = os.path.join(output_dir, filename)
        filtered_df.to_csv(output_file, sep='\t', index=True)
        print(f"Filtered file saved to: {output_file}")

In [ ]:
import os
import pandas as pd
import numpy as np
import scprep as scp

gene_dir = './stnet_heg250'
output_directory_path = './stnet_heg250_normal_smooth'

os.makedirs(output_directory_path, exist_ok=True)

def get_surrounding_spots_average(df):
    averaged_df = pd.DataFrame(index=df.index, columns=df.columns)
    
    for idx in df.index:
        x, y = map(int, idx.split('x'))
        surrounding_indices = [f"{i}x{j}" for i in range(x-1, x+2) for j in range(y-1, y+2)]
        surrounding_data = df.loc[df.index.intersection(surrounding_indices)]
        averaged_df.loc[idx] = surrounding_data.mean(axis=0)
    
    return averaged_df

for filename in os.listdir(gene_dir):
    if filename.endswith('.tsv'):
        file_path = os.path.join(gene_dir, filename)
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        
        normalized_log_transformed_df = scp.transform.log(scp.normalize.library_size_normalize(df))
        averaged_df = get_surrounding_spots_average(normalized_log_transformed_df)
        output_file_path = os.path.join(output_directory_path, f'{filename}')
        averaged_df.to_csv(output_file_path, sep='\t')
        
        print(f"Processed and saved: {output_file_path}")

In [ ]:
import os
import pandas as pd

genes = []

for i in os.listdir(skin_gene_expression):
    path = os.path.join(skin_gene_expression,i)
    print(f"Gene expression file : {path}")
    data = pd.read_csv(path, sep='\t')
    spots = data[data.columns[0]].to_list()
    genes_i = data.columns.tolist()[1:]
    genes.append(genes_i)
    print(spots)
    print(len(spots))
    print(genes_i)
    print(len(genes_i))

In [ ]:
total_genes = set().union(*genes)
lens = len(total_genes)

inter_genes = total_genes.intersection(*genes)
lens2 = len(inter_genes)

In [5]:
gene_expression_sum = {}
for filename in os.listdir(skin_gene_expression):
    if filename.endswith('.tsv'):
        file_path = os.path.join(skin_gene_expression, filename)
        print(f"Processing file: {file_path}")
        df = pd.read_csv(file_path, sep='\t')
        temp_genes = df.columns[1:]
        print(temp_genes)
        for gene in inter_genes:
            total_expression = df[gene].sum()
            if gene in gene_expression_sum:
                gene_expression_sum[gene] += total_expression
            else:
                gene_expression_sum[gene] = total_expression

print(gene_expression_sum)

In [ ]:
sorted_gene_expression = sorted(gene_expression_sum.items(), key=lambda x: x[1], reverse=True)
print("Sorted Gene Expression:")
gene_select_heg_nums = []
with open('skin_top_250_genes.csv', 'w') as f:
    f.write("Gene,Expression\n")
    for gene, expression in sorted_gene_expression[:500]:
        if gene in inter_genes:
            gene_select_heg_nums.append(gene)
            f.write(f"{gene},{expression}\n")
        if len(gene_select_heg_nums) == 250:
            break

print("Top 250 genes have been saved to 'skin_top_250_genes.csv'.")

In [ ]:
output_dir = './skin_top250_heg'
selected_genes_file = 'skin_top_250_genes.csv'
os.makedirs(output_dir, exist_ok=True)
selected_genes_df = pd.read_csv(selected_genes_file)
selected_genes = selected_genes_df['Gene'].tolist() 
for filename in os.listdir(skin_gene_expression):
    if filename.endswith('.tsv'): 
        file_path = os.path.join(skin_gene_expression, filename)
        print(f"Processing file: {file_path}")
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        filtered_df = df[selected_genes]
        output_file = os.path.join(output_dir, filename)
        filtered_df.to_csv(output_file, sep='\t', index=True)
        print(f"Filtered file saved to: {output_file}")

In [ ]:
import os
import pandas as pd
import numpy as np
import scprep as scp

gene_dir = './skin_top250_heg'
output_directory_path = './skin_heg250_normal_smooth'

os.makedirs(output_directory_path, exist_ok=True)

def get_surrounding_spots_average(df):
    averaged_df = pd.DataFrame(index=df.index, columns=df.columns)
    
    for idx in df.index:
        x, y = map(int, idx.split('x'))
        surrounding_indices = [f"{i}x{j}" for i in range(x-1, x+2) for j in range(y-1, y+2)]
        surrounding_data = df.loc[df.index.intersection(surrounding_indices)]
        averaged_df.loc[idx] = surrounding_data.mean(axis=0)
    
    return averaged_df

for filename in os.listdir(gene_dir):
    if filename.endswith('.tsv'):
        file_path = os.path.join(gene_dir, filename)
        df = pd.read_csv(file_path, sep='\t', index_col=0)
        
        normalized_log_transformed_df = scp.transform.log(scp.normalize.library_size_normalize(df))
        averaged_df = get_surrounding_spots_average(normalized_log_transformed_df)
        output_file_path = os.path.join(output_directory_path, f'{filename}')
        averaged_df.to_csv(output_file_path, sep='\t')
        
        print(f"Processed and saved: {output_file_path}")